[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/03_minimal_ai_systems_overview.ipynb)

# 03. Minimal AI systems overview — paper-faithful tiny models

목표는 **크기만 줄이고 원형의 중요한 계산 그래프는 보존**하는 것이다.

공통 attention도 `nn.TransformerEncoderLayer` 하나로 감추지 않고 `Q/K/V → scaled dot-product → mask → softmax → V 가중합`이 실제 구현에 드러나게 만들었다.

- GPT: GPT-2 계열 learned token/position embedding + causal self-attention + pre-norm residual + tied LM head
- ViT: patch embedding + CLS token + learned positional embedding + Transformer encoder
- 2D DiT: fixed 2D sin-cos position + timestep embedding + adaLN-Zero + conditioned final layer
- 3D DiT: 같은 DiT 원리를 3D non-overlapping patch grid로 확장
- VLA: π0/openpi처럼 vision+language prefix와 state+noisy-action suffix가 같은 masked Transformer attention에 참여


In [ ]:
import importlib.util
import sys
import urllib.request

import torch
import torch.nn.functional as F

MODEL_URL = (
    "https://raw.githubusercontent.com/"
    "HisameOgasahara/deep-learning-diagnostics-and-improvement/"
    "main/practice/paper_faithful_tiny_models.py"
)
LOCAL_MODEL_PATH = "/tmp/paper_faithful_tiny_models.py"

urllib.request.urlretrieve(
    MODEL_URL,
    LOCAL_MODEL_PATH,
)

spec = importlib.util.spec_from_file_location(
    "tiny_models",
    LOCAL_MODEL_PATH,
)
tiny_models = importlib.util.module_from_spec(spec)
sys.modules["tiny_models"] = tiny_models
spec.loader.exec_module(tiny_models)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
torch.manual_seed(7)

print("device:", device)
print("torch:", torch.__version__)


## A. Tiny GPT-2-like decoder

각 head는 `Q=XW_Q`, `K=XW_K`, `V=XW_V`를 만들고 `softmax(QKᵀ/√d_h)V`를 계산한다. causal mask 때문에 위치 `i`는 `j>i`인 미래 token을 볼 수 없다.

여기서는 GPT-2 계열의 중요한 구조인 **learned absolute position embedding, pre-norm residual block, causal self-attention, tied LM head**를 유지한다.


In [ ]:
gpt = tiny_models.TinyGPT().to(device)
optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=3e-3,
)

tokens = torch.tensor(
    [[1, 2, 3, 4, 5, 6, 7, 8]],
    device=device,
)

for step in range(4):
    input_tokens = tokens[:, :-1]
    target_tokens = tokens[:, 1:]

    logits = gpt(input_tokens)
    loss = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        target_tokens.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "GPT step",
        step,
        "loss",
        round(loss.item(), 4),
    )

with torch.no_grad():
    _, attention_maps = gpt(
        tokens[:, :-1],
        return_attn=True,
    )

    sequence_length = tokens.size(1) - 1
    future_positions = torch.triu(
        torch.ones(
            sequence_length,
            sequence_length,
            dtype=torch.bool,
            device=device,
        ),
        diagonal=1,
    )

    first_layer_attention = attention_maps[0]
    future_attention_mass = first_layer_attention[
        ...,
        future_positions,
    ].sum()

    print("attention shape:", tuple(first_layer_attention.shape))
    print("future attention mass:", float(future_attention_mass))


## B. Tiny ViT

원 ViT의 계산 사슬을 그대로 줄인다.

`image → non-overlapping patches → linear projection → [CLS] + learned positional embedding → full self-attention encoder → CLS classifier`

이전 버전에서 빠졌던 **위치 정보**를 포함한다.


In [ ]:
vit = tiny_models.TinyViT().to(device)

images = torch.randn(
    4, 3, 16, 16,
    device=device,
)
labels = torch.tensor(
    [0, 1, 2, 1],
    device=device,
)

optimizer = torch.optim.AdamW(
    vit.parameters(),
    lr=3e-3,
)

for step in range(4):
    logits = vit(images)
    loss = F.cross_entropy(logits, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "ViT step",
        step,
        "loss",
        round(loss.item(), 4),
    )

with torch.no_grad():
    _, attention_maps = vit(
        images,
        return_attn=True,
    )
    first_layer_attention = attention_maps[0]

    print(
        "CLS + patch token count:",
        first_layer_attention.shape[-1],
    )
    print(
        "attention shape:",
        tuple(first_layer_attention.shape),
    )


## C. Tiny 2D DiT + Flow Matching

DiT 공식 구조에서 **patch tokenization, fixed 2D sin-cos position, sinusoidal timestep MLP, adaLN-Zero의 shift/scale/gate, conditioned final layer, unpatchify**를 유지한다.

학습 objective만 Flow Matching의 직선 path `x_t=(1-t)x_data+t x_noise`, target velocity `u_t=x_noise-x_data`를 사용한다. 즉 **backbone은 DiT**, **학습 target은 Flow Matching**이다.


In [ ]:
dit2 = tiny_models.Tiny2DDiT().to(device)
optimizer = torch.optim.AdamW(
    dit2.parameters(),
    lr=3e-3,
)

x_data = torch.randn(
    3, 2, 8, 8,
    device=device,
)
x_noise = torch.randn_like(x_data)

for step in range(6):
    t = torch.rand(3, device=device)
    t_broadcast = t[:, None, None, None]

    x_t = (
        (1 - t_broadcast) * x_data
        + t_broadcast * x_noise
    )
    target_velocity = x_noise - x_data

    predicted_velocity = dit2(x_t, t)
    loss = F.mse_loss(
        predicted_velocity,
        target_velocity,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "2D DiT-flow step",
        step,
        "loss",
        round(loss.item(), 4),
    )

with torch.no_grad():
    _, attention_maps = dit2(
        x_data[:1],
        torch.zeros(1, device=device),
        return_attn=True,
    )
    print("attention shape:", tuple(attention_maps[0].shape))


## D. Tiny 3D DiT + Flow

단일 표준 ‘3D DiT 원논문’이라고 가장하지 않는다. 2D DiT의 구조 원리를 3D에 교육용으로 확장해서 **voxel 하나가 아니라 non-overlapping 3D patch 하나를 token으로 만들고 3D sin-cos position을 부여**한다. attention, timestep conditioning, adaLN-Zero는 2D DiT와 같은 원리를 쓴다.


In [ ]:
dit3 = tiny_models.Tiny3DDiT().to(device)
optimizer = torch.optim.AdamW(
    dit3.parameters(),
    lr=3e-3,
)

x_data = torch.randn(
    3, 1, 4, 4, 4,
    device=device,
)
x_noise = torch.randn_like(x_data)

for step in range(6):
    t = torch.rand(3, device=device)
    t_broadcast = t[:, None, None, None, None]

    x_t = (
        (1 - t_broadcast) * x_data
        + t_broadcast * x_noise
    )
    target_velocity = x_noise - x_data

    predicted_velocity = dit3(x_t, t)
    loss = F.mse_loss(
        predicted_velocity,
        target_velocity,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "3D DiT-flow step",
        step,
        "loss",
        round(loss.item(), 4),
    )

patch_size = 2
volume_size = 4
num_patch_tokens = (volume_size // patch_size) ** 3
num_voxels = volume_size ** 3

print("3D patch tokens:", num_patch_tokens)
print("raw voxel count:", num_voxels)


## E. Tiny π0-like VLA Flow Policy

vision/language를 평균낸 뒤 action을 attention 밖에서 더하지 않는다.

- image patch tokens + language tokens → prefix
- robot state + noisy action-horizon tokens → suffix
- prefix와 suffix가 **같은 Transformer attention**에 참여
- prefix는 action suffix를 볼 수 없음
- action suffix는 prefix/state/action block을 볼 수 있음
- noisy action과 timestep은 **action token 자체**에 들어감
- Transformer를 지난 각 action token hidden state에서 velocity를 예측


In [ ]:
policy = tiny_models.TinyVLAFlowPolicy().to(device)
optimizer = torch.optim.AdamW(
    policy.parameters(),
    lr=3e-3,
)

batch_size = 3
vision = torch.randn(
    batch_size, 3, 16, 16,
    device=device,
)
language = torch.tensor(
    [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 9],
    ],
    device=device,
)
state = torch.randn(
    batch_size, 4,
    device=device,
)
action_data = torch.randn(
    batch_size, 4, 4,
    device=device,
)
action_noise = torch.randn_like(action_data)

for step in range(6):
    t = torch.rand(batch_size, device=device)
    t_broadcast = t[:, None, None]

    noisy_action_t = (
        (1 - t_broadcast) * action_data
        + t_broadcast * action_noise
    )
    target_velocity = action_noise - action_data

    predicted_velocity = policy(
        vision,
        language,
        state,
        noisy_action_t,
        t,
    )
    loss = F.mse_loss(
        predicted_velocity,
        target_velocity,
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(
        "VLA-flow step",
        step,
        "loss",
        round(loss.item(), 4),
    )

with torch.no_grad():
    (
        predicted_velocity,
        attention_maps,
        allowed_mask,
    ) = policy(
        vision[:1],
        language[:1],
        state[:1],
        action_noise[:1],
        torch.ones(1, device=device),
        return_attn=True,
    )

    print("velocity shape:", tuple(predicted_velocity.shape))
    print("joint attention shape:", tuple(attention_maps[0].shape))
    print("prefix -> action allowed:", bool(allowed_mask[0, -1]))
    print("action -> prefix allowed:", bool(allowed_mask[-1, 0]))


## References and provenance

**GPT** — Radford et al. (2018), GPT-2 report (2019), Vaswani et al. (2017). learned token/position embedding, causal self-attention, pre-norm residual, tied LM head를 반영했다.

**ViT** — Dosovitskiy et al., *An Image is Worth 16x16 Words*, ICLR 2021. patch projection, CLS token, learned positional embedding, encoder, CLS classifier를 반영했다.

**DiT** — Peebles & Xie, *Scalable Diffusion Models with Transformers*, ICCV 2023 및 공식 `facebookresearch/DiT` 구현. fixed sin-cos position, timestep MLP, adaLN-Zero, conditioned final layer를 반영했다.

**Flow Matching** — Lipman et al., *Flow Matching for Generative Modeling*, ICLR 2023. DiT backbone은 유지하고 학습 target만 velocity regression으로 둔다.

**VLA / π0** — Physical Intelligence, *π0: A Vision-Language-Action Flow Model for General Robot Control* 및 공식 `Physical-Intelligence/openpi` 구현. prefix/suffix attention mask, noisy-action+timestep embedding, action-token flow output을 축소해 보존했다.

구현 본체는 같은 폴더의 `paper_faithful_tiny_models.py`에 두어 notebook에서 구조와 실험 흐름을 읽기 쉽게 분리했다.
